# `train_yield`

Module 1 - Crop YIELD prediction (regression).

Evaluation protocols
  A) Temporal hold-out : train Year <= TRAIN_END_YEAR, test afterwards  (forecasting realism)
  B) Grouped K-fold    : folds split by DISTRICT so a district never appears in train and test
Target: log1p(yield).  Production is never used as a feature (it equals area x yield -> leakage).

> Part of the crop yield & recommendation project. Imported by `run_all.py` (via the notebook loader in `src/__init__.py`) or runnable directly here.

In [ ]:
# Setup: make the project root importable when this notebook is opened directly in Jupyter
import sys
from pathlib import Path
_root = Path.cwd().resolve()
if _root.name == "src":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import json
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.base import clone
from sklearn.ensemble import (GradientBoostingRegressor, RandomForestRegressor,
                              StackingRegressor, VotingRegressor)
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

from src import config as C
from src.features import build_preprocessor
from src.utils import have, savefig, set_seed

warnings.filterwarnings("ignore")
CAT_COLS = ["State", "District", "Crop", "Season"]

In [ ]:
# ----------------------------------------------------------------------------- data
def load_data():
    df = pd.read_csv(C.YIELD_CLEAN)
    df["log_area"] = np.log1p(df["Area"])
    df["y"] = np.log1p(df["Yield"])
    num_cols = ["Year", "log_area"] + (["rainfall_mm"] if "rainfall_mm" in df.columns else [])
    features = num_cols + CAT_COLS
    groups = (df["State"] + "|" + df["District"]).values
    return df, features, num_cols, groups

In [ ]:
# ----------------------------------------------------------------------------- models
def make_models(num_cols, fast=False):
    n = 60 if fast else 300
    raw = {
        "Ridge": ("onehot", Ridge(alpha=1.0)),
        "DecisionTree": ("ordinal", DecisionTreeRegressor(max_depth=14, min_samples_leaf=5, random_state=C.SEED)),
        "KNN": ("onehot", KNeighborsRegressor(n_neighbors=7, weights="distance", n_jobs=-1)),
        "RandomForest": ("ordinal", RandomForestRegressor(n_estimators=n, min_samples_leaf=2,
                                                          n_jobs=-1, random_state=C.SEED)),
        "GradientBoosting": ("ordinal", GradientBoostingRegressor(n_estimators=n, max_depth=5,
                                                                   learning_rate=0.08, subsample=0.8,
                                                                   random_state=C.SEED)),
    }
    if have("xgboost"):
        from xgboost import XGBRegressor
        raw["XGBoost"] = ("ordinal", XGBRegressor(n_estimators=n, learning_rate=0.06, max_depth=8,
                                                  subsample=0.8, colsample_bytree=0.8, n_jobs=-1,
                                                  random_state=C.SEED, verbosity=0))
    if have("lightgbm"):
        from lightgbm import LGBMRegressor
        raw["LightGBM"] = ("ordinal", LGBMRegressor(n_estimators=n, learning_rate=0.06, num_leaves=63,
                                                    subsample=0.8, colsample_bytree=0.8, n_jobs=-1,
                                                    random_state=C.SEED, verbose=-1))
    if have("catboost"):
        from catboost import CatBoostRegressor
        raw["CatBoost"] = ("ordinal", CatBoostRegressor(iterations=n, depth=8, learning_rate=0.08,
                                                        verbose=0, random_seed=C.SEED, thread_count=-1))
    return raw

In [ ]:
def add_ensembles(raw):
    tree_names = [k for k in raw if k not in ("Ridge", "DecisionTree", "KNN")]
    est = [(k, clone(raw[k][1])) for k in tree_names]
    raw["Voting(avg)"] = ("ordinal", VotingRegressor(estimators=est))
    raw["Stacking"] = ("ordinal", StackingRegressor(estimators=est, final_estimator=RidgeCV(), cv=3))
    return raw

In [ ]:
def to_pipelines(raw, num_cols):
    return {k: Pipeline([("prep", build_preprocessor(kind, num_cols, CAT_COLS)), ("model", est)])
            for k, (kind, est) in raw.items()}

In [ ]:
# ----------------------------------------------------------------------------- tuning
SPACES = {
    "RandomForest": {"n_estimators": ("int", 100, 400), "max_depth": ("int", 8, 30),
                     "min_samples_leaf": ("int", 1, 10), "max_features": ("float", 0.4, 1.0)},
    "GradientBoosting": {"n_estimators": ("int", 100, 400), "max_depth": ("int", 3, 9),
                         "learning_rate": ("logfloat", 0.02, 0.2), "subsample": ("float", 0.6, 1.0)},
}

In [ ]:
def _suggest(trial, space):
    out = {}
    for k, (kind, lo, hi) in space.items():
        if kind == "int":
            out[k] = trial.suggest_int(k, lo, hi)
        elif kind == "float":
            out[k] = trial.suggest_float(k, lo, hi)
        else:
            out[k] = trial.suggest_float(k, lo, hi, log=True)
    return out

In [ ]:
def _random_draw(space, rng):
    out = {}
    for k, (kind, lo, hi) in space.items():
        if kind == "int":
            out[k] = int(rng.integers(lo, hi + 1))
        elif kind == "float":
            out[k] = float(rng.uniform(lo, hi))
        else:
            out[k] = float(np.exp(rng.uniform(np.log(lo), np.log(hi))))
    return out

In [ ]:
def tune(name, pipe, X, y, groups, n_trials):
    """Hyper-parameter search with GROUPED CV on the training portion only.
    Uses Optuna (TPE) if installed, otherwise random search."""
    space = SPACES[name]
    folds = list(GroupKFold(3).split(X, y, groups))

    def score(params):
        p = clone(pipe).set_params(**{f"model__{k}": v for k, v in params.items()})
        return float(np.mean([r2_score(y.iloc[te], p.fit(X.iloc[tr], y.iloc[tr]).predict(X.iloc[te]))
                              for tr, te in folds]))

    if have("optuna"):
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=C.SEED))
        study.optimize(lambda t: score(_suggest(t, space)), n_trials=n_trials)
        return study.best_params, study.best_value
    rng = np.random.default_rng(C.SEED)
    best_p, best_s = None, -np.inf
    for _ in range(n_trials):
        p = _random_draw(space, rng)
        s = score(p)
        if s > best_s:
            best_p, best_s = p, s
    return best_p, best_s

In [ ]:
# ----------------------------------------------------------------------------- metrics
def reg_metrics(y_log, p_log):
    p_log = np.clip(p_log, 0, None)
    y, p = np.expm1(y_log), np.expm1(p_log)
    return {"R2_log": r2_score(y_log, p_log),
            "R2": r2_score(y, p),
            "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "MAE": mean_absolute_error(y, p)}

In [ ]:
# ----------------------------------------------------------------------------- main
def main(fast=False, do_tune=False):
    print("[2/4] Yield prediction")
    set_seed()
    df, features, num_cols, groups = load_data()
    X, y = df[features], df["y"]
    print(f"  features: {features}")

    raw = make_models(num_cols, fast)
    pipes = to_pipelines(raw, num_cols)

    train_mask = (df["Year"] <= C.TRAIN_END_YEAR).values
    Xtr, ytr, Xte, yte = X[train_mask], y[train_mask], X[~train_mask], y[~train_mask]
    print(f"  temporal split: train={len(Xtr):,}  test={len(Xte):,}")

    if do_tune:
        n_trials = 3 if fast else C.N_TRIALS
        for name in ["RandomForest", "GradientBoosting"]:
            t0 = time.time()
            best, val = tune(name, pipes[name], Xtr, ytr, groups[train_mask], n_trials)
            print(f"  tuned {name}: grouped-CV R2_log={val:.4f} ({time.time()-t0:.0f}s) {best}")
            raw[f"{name}(tuned)"] = ("ordinal", clone(raw[name][1]).set_params(**best))
        pipes = to_pipelines(raw, num_cols)

    # ensembles built from the (possibly tuned) tree models
    tree_only = {k: v for k, v in raw.items() if k not in ("Ridge", "DecisionTree", "KNN") and "(tuned)" not in k}
    for base in ("RandomForest", "GradientBoosting"):          # ensembles use tuned params if available
        if f"{base}(tuned)" in raw:
            tree_only[base] = raw[f"{base}(tuned)"]
    for k, v in add_ensembles(dict(tree_only)).items():
        if k in ("Voting(avg)", "Stacking"):
            raw[k] = v
    pipes = to_pipelines(raw, num_cols)

    cv = list(GroupKFold(C.CV_FOLDS).split(X, y, groups))
    rows, fold_scores, temporal_pred = [], {}, {}
    for name, pipe in pipes.items():
        t0 = time.time()
        m = clone(pipe).fit(Xtr, ytr)
        pt = m.predict(Xte)
        temporal_pred[name] = pt
        met = reg_metrics(yte.values, pt)
        scores = []
        for tr, te in cv:
            scores.append(r2_score(y.iloc[te], np.clip(clone(pipe).fit(X.iloc[tr], y.iloc[tr]).predict(X.iloc[te]), 0, None)))
        fold_scores[name] = scores
        rows.append({"Model": name, **{f"Temporal_{k}": v for k, v in met.items()},
                     "GroupCV_R2_log_mean": np.mean(scores), "GroupCV_R2_log_std": np.std(scores),
                     "Time_s": time.time() - t0})
        print(f"  {name:<22} temporal R2_log={met['R2_log']:.4f} | grouped-CV R2_log={np.mean(scores):.4f}"
              f" +/- {np.std(scores):.4f} ({time.time()-t0:.0f}s)")

    res = pd.DataFrame(rows).sort_values("GroupCV_R2_log_mean", ascending=False).reset_index(drop=True)
    res.to_csv(C.REPORTS / "yield_results.csv", index=False)
    best_name = res.loc[0, "Model"]
    print(f"  BEST (grouped CV): {best_name}")

    # ---- significance tests: best model vs every other (paired over CV folds)
    sig = []
    for name in res["Model"][1:]:
        a, b = np.array(fold_scores[best_name]), np.array(fold_scores[name])
        t_p = stats.ttest_rel(a, b).pvalue
        try:
            w_p = stats.wilcoxon(a, b).pvalue
        except ValueError:
            w_p = np.nan
        sig.append({"Best": best_name, "Compared_to": name, "mean_R2_diff": float((a - b).mean()),
                    "paired_t_p": t_p, "wilcoxon_p": w_p})
    pd.DataFrame(sig).to_csv(C.REPORTS / "yield_significance.csv", index=False)

    # ---- figures
    plt.figure(figsize=(9, 4.8))
    sns.barplot(data=res, y="Model", x="GroupCV_R2_log_mean", color="seagreen", xerr=res["GroupCV_R2_log_std"])
    plt.xlim(max(0, res["GroupCV_R2_log_mean"].min() - 0.05), 1.0)
    plt.xlabel("Grouped-CV R2 (log yield)"); plt.title("Yield models - district-grouped CV")
    savefig("yield_model_comparison.png")

    p_best = np.expm1(np.clip(temporal_pred[best_name], 0, None))
    t_best = np.expm1(yte.values)
    plt.figure(figsize=(5.5, 5.5))
    plt.scatter(t_best, p_best, s=5, alpha=0.3)
    lim = [max(1e-2, min(t_best.min(), p_best.min())), max(t_best.max(), p_best.max())]
    plt.plot(lim, lim, "r--", lw=1); plt.xscale("log"); plt.yscale("log")
    plt.xlabel("Actual yield"); plt.ylabel("Predicted yield"); plt.title(f"{best_name} - temporal test set")
    savefig("yield_pred_vs_actual.png")

    # ---- refit best on ALL data & save for the app
    final = clone(pipes[best_name]).fit(X, y)
    joblib.dump(final, C.MODELS / "yield_best.joblib")

    meta = {
        "best_model": best_name, "features": features, "num_cols": num_cols, "cat_cols": CAT_COLS,
        "has_rain": "rainfall_mm" in df.columns,
        "year_min": int(df["Year"].min()), "year_max": int(df["Year"].max()),
        "geo": {s: sorted(g["District"].unique().tolist()) for s, g in df.groupby("State")},
        "state_crops": {s: sorted(g["Crop"].unique().tolist()) for s, g in df.groupby("State")},
        "crop_seasons": {c: sorted(g["Season"].unique().tolist()) for c, g in df.groupby("Crop")},
        "median_area": {c: float(g["Area"].median()) for c, g in df.groupby("Crop")},
        "metrics": res.iloc[0].to_dict(),
    }
    if meta["has_rain"]:
        meta["state_rain"] = {s: float(v) for s, v in df.groupby("State")["rainfall_mm"].mean().items()}
    (C.MODELS / "yield_meta.json").write_text(json.dumps(meta, indent=2, default=float))
    print("  saved models/yield_best.joblib + yield_meta.json")
    return res

In [ ]:
if __name__ == "__main__":
    main()